In [ ]:
cd ../..

## Feature Ideas

### Basic Features
- Driver features:
    - `driver_ref`: Categorical identifier for driver
    - `driver_age`
    - `driver_nationality`
- Driver (current) standings:
    - `driver_position`
    - `driver_points`
    - `driver_wins`
    - `driver_points_lag`: Point difference with driver ahead
    - `driver_points_lead`: Point difference with driver behind
    - `days_since_last_race`
- Constructor features:
    - `constructor_ref`: Categorical identifier for constructor
    - `constructor_nationality`
- Constructor (current) standings:
    - `constructor_position`
    - `constructor_points`
    - `constructor_wins`
    - `constructor_points_lag`: Point difference with constructor ahead
    - `constructor_points_lead`: Point difference with constructor behind
- Driver contribution:
    - `position_diff`: Difference in position of driver and constructor
    - `points_ratio`: How many of the constructor's points come from this driver
    - `wins_ratio`: How many of constructor's wins come from this driver


### Past Race Features
- For each of the past 3 races:
    - `driver_position`
    - `driver_points`
    - `driver_grid`
    - `position_gain`
    - `driver_rank`
    - `pct_laps_completed`
    - `dnf`
    - `constructor_points`

### Upcoming Race Features
- For the upcoming 3 races:
    - `round`
    - `circuitId`: Categorical identifier for the circuit

In [ ]:
import duckdb
from torch_frame.utils import infer_df_stype

import utils

conn = duckdb.connect('f1/f1.db')
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaycon=False
%config SqlMagic.autopandas=True

In [ ]:
with open('f1/driver-top3/feats.sql', 'r') as f:
    # run once with train_labels and once with val_labels
    template = f.read()

# create train, val and test features
# takes 1 - 5 mins
for s in ['train', 'val', 'test']:
    print(f'Creating {s} table')
    query = utils.render_jinja_sql(template, dict(set=s, subsample=0))
    conn.sql(query)
    print(f'{s} table created')

In [ ]:
utils.validate_feature_tables('driver-top3', conn)

In [ ]:
%%sql train_df <<
from driver_top3_train_feats

In [ ]:
infer_df_stype(train_df)

In [ ]:
utils.feature_summary_df(train_df, 'qualifying')

## Feature Importances

In [ ]:
import shap
from torch_frame import TaskType, stype
from torch_frame.data import Dataset
from torch_frame.gbdt import LightGBM

from inferred_stypes import task_to_stypes
from train_gbdt import TASK_PARAMS

TASK = 'rel-f1-driver-top3'

task_params = TASK_PARAMS[TASK]

In [ ]:
%%sql train_df <<
select * from driver_top3_train_feats;

In [ ]:
%%sql val_df <<
select * from driver_top3_val_feats;

In [ ]:
train_dset = Dataset(
    train_df,
    col_to_stype=task_to_stypes[TASK],
    target_col=task_params['target_col'],
).materialize()
val_tf = train_dset.convert_to_tensor_frame(val_df)

In [ ]:
gbdt = LightGBM(task_type=task_params['task_type'])
gbdt.load(f'models/{TASK}_lgbm.json')
pred = gbdt.predict(tf_test=val_tf).numpy()

In [ ]:
shap.initjs()
explainer = shap.TreeExplainer(gbdt.model)

val_arr, _, _ = gbdt._to_lightgbm_input(val_tf[:10_000])
shap_values = explainer.shap_values(val_arr, pred[:10_000])

# TODO verify
feat_names = val_tf.col_names_dict.get(stype.categorical, []) + val_tf.col_names_dict[stype.numerical]

shap.summary_plot(shap_values, val_arr, plot_type='violin', max_display=30, feature_names=feat_names)

In [ ]:
conn.close()